# Основы машинного обучения: классификация с помощью sklearn

**Тема**: бинарная классификация — предсказание выживаемости персонажей «Игры Престолов».

В этом конспекте разберём полный pipeline ML-задачи:
1. Загрузка и анализ данных (EDA)
2. Предобработка и feature engineering
3. Обучение и сравнение моделей
4. Оценка качества и подбор гиперпараметров

Конспект **самодостаточный** — весь код можно запустить независимо от solution.ipynb.

## 1. Ключевые концепции

### Задача классификации
Классификация — задача ML с учителем, где модель предсказывает **категорию** для каждого объекта. Бинарная классификация — два класса (0/1).

### Метрика Accuracy
$$\text{Accuracy} = \frac{\text{Число верных предсказаний}}{\text{Общее число предсказаний}}$$

Простая метрика, но обманчива при дисбалансе: если 78% персонажей живы, модель-«заглушка» (`predict = 1` для всех) даст Accuracy = 0.78.

### Data Leakage (утечка данных)
Когда в признаки попадает информация, недоступная на момент предсказания. Например, `dateOfDeath` напрямую выдаёт ответ — если заполнена, персонаж мёртв.

### Pipeline ML
1. Загрузка и анализ данных (EDA)
2. Предобработка (очистка, заполнение пропусков, кодирование)
3. Выбор и обучение моделей
4. Оценка качества (кросс-валидация)
5. Применение к тестовым данным

## 2. Загрузка и анализ данных

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gdown
import warnings

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
%matplotlib inline

RANDOM_STATE = 42
print('Библиотеки импортированы')

In [ ]:
# Скачиваем данные
os.makedirs('data', exist_ok=True)

gdown.download(id='1XL0VTygpZj-ZAuTNRBgApZTPQyNDnT-v', output='data/game_of_thrones_train.csv', quiet=True)
gdown.download(id='1h99toeF7lZ2I3iJwehgKO-QQmDaOe_O3', output='data/game_of_thrones_test.csv', quiet=True)

# Загружаем в DataFrame, используя S.No как индекс
data = pd.read_csv('data/game_of_thrones_train.csv', index_col='S.No')

print(f'Размер датасета: {data.shape}')
print(f'Целевая переменная isAlive: {data["isAlive"].value_counts().to_dict()}')
print(f'Доля живых: {data["isAlive"].mean():.3f}')
data.head(3)

### Анализ пропущенных значений

Пропуски — одна из главных проблем этого датасета. Удалять строки нельзя (потеряем 80%+ данных), нужно обрабатывать по-умному.

In [ ]:
# Анализ пропусков — полезный паттерн для любого датасета
missing = data.isnull().sum()
missing_pct = (missing / len(data) * 100).round(1)

missing_report = pd.DataFrame({
    'Пропуски': missing,
    '%': missing_pct
}).query('Пропуски > 0').sort_values('%', ascending=False)

print('Признаки с пропусками:')
print(missing_report)
print(f'\nВывод: mother/father/heir — 98%+ пропусков (удаляем),')
print(f'age/dateOfBirth — 82% (создаём индикатор), culture — 69% (группируем)')

## 3. Предобработка и Feature Engineering

### 3.1. Логарифмическое преобразование скошенного признака

Когда распределение признака сильно скошено (большинство значений близко к 0, единичные — далеко), **логарифм** приближает его к нормальному. Это помогает линейным моделям.

In [ ]:
# Демонстрация: распределение popularity до и после логарифма
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

# До
data['popularity'].hist(bins=50, ax=axes[0], color='salmon')
axes[0].set_title('popularity (исходное)')
axes[0].set_xlabel('Значение')

# После: формула np.log10(x * M + 1), где M=100
M = 100
pop_log = np.log10(data['popularity'] * M + 1)
pop_log.hist(bins=50, ax=axes[1], color='skyblue')
axes[1].set_title('popularity (после log10)')
axes[1].set_xlabel('Значение')

plt.tight_layout()
plt.show()

print('Формула: np.log10(popularity * 100 + 1)')
print('+1 нужен, чтобы log(0) не давал -inf')

### 3.2. Обработка пропусков: значение + индикатор

Когда пропусков много, нельзя просто заполнить средним — потеряем информацию. Паттерн «значение + индикатор»:
- `age_value` = возраст, если известен, иначе 0
- `age_no_data` = 1, если возраст не указан, иначе 0

Это позволяет модели учитывать **и значение, и факт его отсутствия**.

In [ ]:
# Демонстрация паттерна «значение + индикатор»
print(f'Пропусков в age: {data["age"].isna().sum()} из {len(data)} ({data["age"].isna().mean()*100:.1f}%)')
print()

# Создаём два признака
age_value = data['age'].fillna(0)
age_no_data = data['age'].isna().astype(int)

# Проверяем: сам факт пропуска информативен!
print('Доля живых среди персонажей:')
print(f'  С указанным возрастом (age_no_data=0): {data.loc[age_no_data==0, "isAlive"].mean():.3f}')
print(f'  Без указанного возраста (age_no_data=1): {data.loc[age_no_data==1, "isAlive"].mean():.3f}')
print()
print('Вывод: персонажи без указанного возраста выживают чаще!')
print('Если бы мы просто заполнили NaN средним, эта информация была бы потеряна.')

### 3.3. Группировка категориального признака и One-Hot Encoding

Когда у категориального признака слишком много уникальных значений (51 культура), one-hot encoding создаст слишком много колонок. Решение — **сгруппировать** похожие значения через словарь маппинга.

In [ ]:
# Словарь группировки культур: 51 значение -> 11 групп
cultures_grouped = {
    'Old Nations': ['valyrian', 'first men', 'andal', 'andals', 'rhoynar'],
    'the North': ['northmen', 'northern mountain clans', 'crannogmen'],
    'the Iron Islands': ['ironborn', 'ironborn', 'ironmen'],
    'the Mountain and the Vale': ['valemen', 'vale', 'vale mountain clans', 'sistermen'],
    'the Isles and Rivers': ['riverlands', 'rivermen'],
    'the Rock': ['westerman', 'westermen', 'westerlands'],
    'the Stormlands': ['stormlander', 'stormlands'],
    'the Reach': ['reach', 'reachmen', 'the reach'],
    'Dorne': ['dornish', 'dornishmen', 'dorne'],
    'Essos Nations': ['astapor', 'astapori', 'braavosi', 'braavos', 'tyroshi', 'lysene', 'lyseni',
                      'myrish', 'pentoshi', 'qartheen', 'qarth', 'dothraki',
                      'lhazarene', 'lhazareen', 'meereen', 'meereenese',
                      'norvoshi', 'qohor', 'summer isles', 'summer islands',
                      'summer islander', 'asshai', "asshai'i", 'norvos', 'ghiscari', 'ghiscaricari'],
    'Other Nations': ['ibbenese', 'westeros', 'free folk', 'wildling', 'wildlings', 'naathi']
}

# Инвертируем: {значение: группа} для удобного маппинга
cultures_grouped_inverted = {}
for group, values in cultures_grouped.items():
    for v in values:
        cultures_grouped_inverted[v] = group

print(f'Маппинг: {len(cultures_grouped_inverted)} значений -> {len(cultures_grouped)} групп')

# Применяем маппинг
culture_mapped = data['culture'].str.lower().str.strip().map(cultures_grouped_inverted).fillna('culture_no_data')

print(f'\nРаспределение групп:')
print(culture_mapped.value_counts())

# One-hot encoding: каждая группа -> отдельная колонка 0/1
culture_dummies = pd.get_dummies(culture_mapped, prefix='cult', dtype=int)
print(f'\nСоздано {culture_dummies.shape[1]} бинарных колонок')
culture_dummies.head(3)

### 3.4. Функция предобработки

Оформление предобработки как **функции** — критически важный паттерн. Это гарантирует, что train и test обрабатываются идентично.

In [ ]:
def preprocess(df, is_test=False):
    """
    Единый pipeline предобработки для train и test данных.
    
    Шаги:
    1. Исправление ошибок в тестовых данных
    2. Удаление data leakage и бесполезных признаков
    3. Логарифмирование popularity
    4. Бинаризация numDeadRelations
    5. Разделение age на значение + индикатор
    6. Обработка isAliveSpouse аналогично age
    7. Группировка culture + one-hot encoding
    """
    d = df.copy()
    
    # Исправление ошибок в тестовых данных
    if is_test:
        if 1685 in d.index:
            d.loc[1685, 'dateOfBirth'] = 278.
            d.loc[1685, 'age'] = 0.
        if 1869 in d.index:
            d.loc[1869, 'dateOfBirth'] = 299.
            d.loc[1869, 'age'] = 0.
    
    # Удаление data leakage
    for col in ['dateOfDeath', 'DateoFdeath']:
        if col in d.columns:
            d.drop(columns=[col], inplace=True)
    
    # Удаление текстовых / высококардинальных / разреженных признаков
    drop_cols = ['name', 'title', 'house', 'spouse', 'father', 'mother', 'heir',
                 'isAliveMother', 'isAliveFather', 'isAliveHeir']
    d.drop(columns=[c for c in drop_cols if c in d.columns], inplace=True)
    
    # Popularity: log-трансформация
    d['popularity_log'] = np.log10(d['popularity'] * 100 + 1)
    d.drop(columns=['popularity'], inplace=True)
    
    # numDeadRelations: бинарный признак
    d['boolDeadRelations'] = (d['numDeadRelations'] > 0).astype(int)
    
    # Age: значение + индикатор пропуска
    d['age_value'] = d['age'].fillna(0)
    d['age_no_data'] = d['age'].isna().astype(int)
    d.drop(columns=['age'], inplace=True)
    
    # dateOfBirth коллинеарен с age_no_data — удаляем
    if 'dateOfBirth' in d.columns:
        d.drop(columns=['dateOfBirth'], inplace=True)
    
    # isAliveSpouse: значение + индикатор
    d['spouse_alive_value'] = d['isAliveSpouse'].fillna(0).astype(int)
    d['spouse_alive_known'] = d['isAliveSpouse'].notna().astype(int)
    d.drop(columns=['isAliveSpouse'], inplace=True)
    
    # Culture: группировка + one-hot
    d['culture_grouped'] = d['culture'].str.lower().str.strip().map(cultures_grouped_inverted)
    d['culture_grouped'] = d['culture_grouped'].fillna('culture_no_data')
    d.drop(columns=['culture'], inplace=True)
    culture_oh = pd.get_dummies(d['culture_grouped'], prefix='cult', dtype=int)
    d = pd.concat([d.drop(columns=['culture_grouped']), culture_oh], axis=1)
    
    return d

# Применяем
train_proc = preprocess(data, is_test=False)
print(f'Признаков после обработки: {train_proc.shape[1] - 1} (+ isAlive)')
print(f'NaN: {train_proc.isnull().sum().sum()}')

### 3.5. Корреляция признаков с целевой переменной

Полезно посмотреть, какие признаки сильнее всего связаны с целевой переменной. Это помогает понять данные и проверить на data leakage (если корреляция > 0.9 — подозрительно).

In [ ]:
# Корреляция с isAlive
corr = train_proc.corr()['isAlive'].drop('isAlive').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
corr.head(12).plot(
    kind='barh', ax=ax,
    color=['salmon' if v < 0 else 'skyblue' for v in corr.head(12)]
)
ax.set_title('Топ-12 признаков по корреляции с isAlive')
ax.set_xlabel('Корреляция Пирсона')
plt.tight_layout()
plt.show()

print('Наблюдения:')
print('  - boolDeadRelations: мёртвые родственники -> меньше шансов выжить')
print('  - book4: появление в 4-й книге -> персонаж скорее жив')
print('  - popularity_log: популярные персонажи чаще погибают')

## 4. Обучение моделей

### 4.1. Подготовка данных для обучения

In [ ]:
# Формируем X и y
train_features = [c for c in train_proc.columns if c != 'isAlive']
X = train_proc[train_features].values
y = train_proc['isAlive'].values

# Разделение на train и validation с стратификацией
# stratify=y сохраняет пропорции классов в обоих подмножествах
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'X_train: {X_train.shape}, X_val: {X_val.shape}')
print(f'Баланс классов train: {np.mean(y_train):.3f} (живых)')
print(f'Баланс классов val:   {np.mean(y_val):.3f} (живых)')

### 4.2. Обучение и сравнение моделей

Обучим 4 модели и сравним их Accuracy на валидации.

**Важно**: для LogReg и KNN нужен `StandardScaler` (масштабирование), поэтому используем `Pipeline`.

In [ ]:
# Модель 1: Logistic Regression (с Pipeline для масштабирования)
# Pipeline гарантирует, что scaler обучается только на train
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, solver='liblinear'))
])
pipe_lr.fit(X_train, y_train)
acc_lr = accuracy_score(y_val, pipe_lr.predict(X_val))

# Модель 2: Random Forest (масштабирование не нужно)
rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)
acc_rf = accuracy_score(y_val, rf.predict(X_val))

# Модель 3: KNN (с Pipeline)
pipe_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=15))
])
pipe_knn.fit(X_train, y_train)
acc_knn = accuracy_score(y_val, pipe_knn.predict(X_val))

# Модель 4: Decision Tree (масштабирование не нужно)
dt = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)
dt.fit(X_train, y_train)
acc_dt = accuracy_score(y_val, dt.predict(X_val))

# Сравнение
results = pd.DataFrame({
    'Модель': ['Logistic Regression', 'Random Forest', 'KNN (k=15)', 'Decision Tree'],
    'Accuracy (val)': [acc_lr, acc_rf, acc_knn, acc_dt]
}).sort_values('Accuracy (val)', ascending=False).reset_index(drop=True)

print(results.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 3))
sns.barplot(data=results, x='Accuracy (val)', y='Модель', ax=ax, palette='viridis')
ax.set_title('Сравнение моделей (Accuracy на валидации)')
ax.set_xlim(0.7, 0.85)
plt.tight_layout()
plt.show()

## 5. Подбор гиперпараметров (GridSearchCV)

### Зачем нужна кросс-валидация?

Одно разделение train/val может быть «удачным» или «неудачным». **K-fold Cross-Validation** (обычно K=10):
1. Делим данные на K частей
2. K раз обучаем модель на K-1 частях, тестируем на оставшейся
3. Усредняем результат

`GridSearchCV` = кросс-валидация + перебор всех комбинаций гиперпараметров.

In [ ]:
# GridSearchCV для Logistic Regression (с Pipeline!)
# Обратите внимание: параметры модели в pipeline указываются через шаг__параметр
pipe_lr_cv = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, solver='liblinear'))
])

param_grid_lr = {
    'lr__C': [0.001, 0.01, 0.05, 0.1, 0.3, 0.5, 1.0, 5.0, 10.0],
    'lr__penalty': ['l1', 'l2'],
}

grid_lr = GridSearchCV(pipe_lr_cv, param_grid_lr, cv=10, scoring='accuracy', n_jobs=-1)
grid_lr.fit(X, y)

print(f'LogReg лучшие параметры: {grid_lr.best_params_}')
print(f'LogReg лучший CV-10 Accuracy: {grid_lr.best_score_:.4f}')

In [ ]:
# GridSearchCV для Random Forest
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 5, 6, 7, 8],
    'min_samples_leaf': [3, 5, 10]
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid_rf, cv=10, scoring='accuracy', n_jobs=-1
)
grid_rf.fit(X, y)

print(f'RF лучшие параметры: {grid_rf.best_params_}')
print(f'RF лучший CV-10 Accuracy: {grid_rf.best_score_:.4f}')

In [ ]:
# Сводная таблица результатов кросс-валидации
cv_results = pd.DataFrame({
    'Модель': ['Logistic Regression', 'Random Forest'],
    'Лучшие параметры': [str(grid_lr.best_params_), str(grid_rf.best_params_)],
    'CV-10 Accuracy': [grid_lr.best_score_, grid_rf.best_score_]
}).sort_values('CV-10 Accuracy', ascending=False).reset_index(drop=True)

print('=== Результаты GridSearchCV (10-fold CV) ===')
print(cv_results.to_string(index=False))

best_model = grid_rf.best_estimator_ if grid_rf.best_score_ > grid_lr.best_score_ else grid_lr.best_estimator_
best_name = 'Random Forest' if grid_rf.best_score_ > grid_lr.best_score_ else 'Logistic Regression'
print(f'\nЛучшая модель: {best_name}')

## 6. Оценка качества

### Classification Report и Confusion Matrix

Accuracy — не единственная метрика. `classification_report` показывает **precision**, **recall** и **f1-score** для каждого класса, а **confusion matrix** — сколько объектов правильно и неправильно классифицировано.

In [ ]:
# Оценка лучшей модели на валидации
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_val)

print(f'Accuracy: {accuracy_score(y_val, y_pred):.4f}')
print()
print('Classification Report:')
print(classification_report(y_val, y_pred, target_names=['Dead (0)', 'Alive (1)']))

# Confusion Matrix
cm = confusion_matrix(y_val, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Dead', 'Alive'], yticklabels=['Dead', 'Alive'])
ax.set_xlabel('Предсказание')
ax.set_ylabel('Истинное значение')
ax.set_title(f'Confusion Matrix ({best_name})')
plt.tight_layout()
plt.show()

print('Как читать confusion matrix:')
print('  - Диагональ = верные предсказания')
print('  - Вне диагонали = ошибки')
print(f'  - Верхний правый угол: модель сказала "жив", а персонаж мёртв (False Positive)')
print(f'  - Нижний левый угол: модель сказала "мёртв", а персонаж жив (False Negative)')

### Важность признаков (Feature Importance)

Random Forest позволяет оценить, какие признаки модель считает самыми полезными для предсказания.

In [ ]:
# Feature importance (доступна для деревьев и ансамблей)
if hasattr(best_model, 'feature_importances_'):
    feat_imp = pd.Series(best_model.feature_importances_, index=train_features).sort_values(ascending=False)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    feat_imp.head(15).plot(kind='barh', ax=ax, color='teal')
    ax.set_title(f'Топ-15 признаков по важности ({best_name})')
    ax.set_xlabel('Feature Importance')
    plt.tight_layout()
    plt.show()
    
    print('Топ-5 признаков:')
    for i, (feat, imp) in enumerate(feat_imp.head(5).items(), 1):
        print(f'  {i}. {feat}: {imp:.4f}')
else:
    print('Модель не поддерживает feature_importances_')

## 7. Попробуй сам!

### Задание 1: Эксперимент с признаками
Попробуй добавить или удалить признаки и посмотри, как изменится качество.

**Подсказка**: попробуй убрать one-hot колонки культуры — может, без них будет лучше?

In [ ]:
# Задание 1: убери колонки культуры и сравни результат
# Подсказка: отфильтруй train_features, убрав колонки начинающиеся с 'cult_'

# features_no_culture = [f for f in train_features if not f.startswith('cult_')]
# X_no_culture = train_proc[features_no_culture].values
# score = cross_val_score(
#     RandomForestClassifier(n_estimators=100, max_depth=7, random_state=42),
#     X_no_culture, y, cv=10, scoring='accuracy'
# ).mean()
# print(f'Accuracy без культуры: {score:.4f}')

# Твой код здесь:


### Задание 2: Другой коэффициент M для логарифма popularity
В формуле `np.log10(popularity * M + 1)` мы использовали M=100. Попробуй другие значения (10, 1000) и оцени влияние на качество.

**Подсказка**: используй `cross_val_score` для честного сравнения.

In [ ]:
# Задание 2: сравни разные значения M
# Подсказка: пересоздай признак popularity_log с другим M,
# обнови X и запусти cross_val_score

# for M_test in [10, 100, 1000]:
#     train_proc_copy = train_proc.copy()
#     train_proc_copy['popularity_log'] = np.log10(data['popularity'] * M_test + 1)
#     X_test_m = train_proc_copy[train_features].values
#     score = cross_val_score(
#         RandomForestClassifier(n_estimators=100, max_depth=7, random_state=42),
#         X_test_m, y, cv=10, scoring='accuracy'
#     ).mean()
#     print(f'M={M_test:>5d}: CV Accuracy = {score:.4f}')

# Твой код здесь:


### Задание 3: Добавь ещё одну модель
Попробуй обучить `AdaBoostClassifier` и сравни с остальными через `cross_val_score`.

**Подсказка**: `from sklearn.ensemble import AdaBoostClassifier`

In [ ]:
# Задание 3: обучи AdaBoost и сравни
# from sklearn.ensemble import AdaBoostClassifier
# 
# ada = AdaBoostClassifier(n_estimators=100, random_state=42)
# ada_score = cross_val_score(ada, X, y, cv=10, scoring='accuracy').mean()
# print(f'AdaBoost CV-10 Accuracy: {ada_score:.4f}')

# Твой код здесь:


## 8. Уроки и выводы

1. **Data leakage** — признак `dateOfDeath` почти полностью дублирует target. Всегда проверяйте, нет ли признаков, которые «подсматривают» ответ.

2. **Пропуски != мусор** — сам факт отсутствия данных может быть информативным. Признак `age_no_data` оказался полезным: персонажи без указанного возраста чаще живы.

3. **Feature engineering важнее выбора модели** — разница между моделями ~1%, а правильная обработка признаков критична для хорошего результата.

4. **Группировка категорий** — при большом количестве уникальных значений группировка снижает размерность и уменьшает шум. 51 культура -> 11 групп.

5. **Масштабирование** — обязательно для LogReg и KNN. Для деревьев — не нужно.

6. **Pipeline** — объединяет scaler + модель, предотвращая утечку данных при кросс-валидации.

7. **GridSearchCV + 10-fold CV** — надёжнее одного train/val split для оценки и подбора гиперпараметров.

8. **Одинаковый preprocessing для train и test** — оформляйте как функцию!

9. **`random_state=42`** — для воспроизводимости всех случайных операций.

### Ссылки на документацию
- [sklearn.pipeline.Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)
- [sklearn.model_selection.GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)
- [sklearn.ensemble.RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
- [sklearn.linear_model.LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
- [pandas.get_dummies](https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html)